# Voicebox + Qwen3-TTS for EPUB Player — free Kaggle GPU

This notebook runs the actual open-source `jamiepine/voicebox` backend on a Kaggle GPU and keeps it available while you switch back to the EPUB reader on iPhone.

Before running:
1. In Kaggle notebook settings, turn **Internet ON**.
2. Choose a **GPU accelerator** (P100 or T4 are both fine).
3. Tap **Run All**.
4. Keep the final cell running. It intentionally stays alive and prints a heartbeat so Kaggle keeps the server session active in the cloud.

Kaggle GPU sessions are temporary and use your Kaggle GPU quota, but unlike Colab on iPhone they can continue running when the browser is backgrounded.


In [ ]:
import os, sys, subprocess, pathlib, shutil, sysconfig, urllib.request

# Verify internet early so the failure message is useful on Kaggle.
try:
    urllib.request.urlopen('https://github.com', timeout=10).read(1)
except Exception as e:
    raise RuntimeError('Kaggle Internet is OFF. Open notebook Settings and turn Internet ON, then Run All again.') from e

try:
    import torch
    print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
except Exception as e:
    raise RuntimeError('PyTorch/GPU check failed') from e
if not torch.cuda.is_available():
    raise RuntimeError('No GPU attached. In Kaggle Settings choose a GPU accelerator, then Run All again.')


In [ ]:
# Install Voicebox in an isolated Python 3.11 environment.
import os, sys, subprocess, pathlib, shutil, sysconfig

ROOT = pathlib.Path('/kaggle/working/voicebox')
VENV = pathlib.Path('/kaggle/working/voicebox-py311')
PIN = '51f49dea198384b4eb6087b72c17057c6eb1c1cd'

def run(cmd, cwd=None, env=None):
    print('+', ' '.join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), cwd=cwd, env=env, check=True)

if not ROOT.exists():
    run(['git','clone','https://github.com/jamiepine/voicebox.git',str(ROOT)])
run(['git','-C',str(ROOT),'fetch','--depth','1','origin',PIN])
run(['git','-C',str(ROOT),'checkout','--detach',PIN])
run(['git','-C',str(ROOT),'reset','--hard',PIN])
run(['git','-C',str(ROOT),'clean','-fd'])

# Install uv without assuming a hard-coded executable path.
run([sys.executable,'-m','pip','install','-q','--upgrade','uv'])
candidates = [
    shutil.which('uv'),
    str(pathlib.Path(sysconfig.get_path('scripts')) / 'uv'),
    str(pathlib.Path.home() / '.local' / 'bin' / 'uv'),
]
UV = next((p for p in candidates if p and pathlib.Path(p).exists()), None)
if not UV:
    raise RuntimeError('uv installed but its executable could not be located')
print('uv:', UV)
run([UV,'python','install','3.11'])

py = VENV / 'bin' / 'python'
rebuild = True
if py.exists():
    try:
        version = subprocess.check_output([str(py),'-c',"import sys; print(f'{sys.version_info.major}.{sys.version_info.minor}')"], text=True).strip()
        rebuild = version != '3.11'
    except Exception:
        rebuild = True
if rebuild and VENV.exists():
    shutil.rmtree(VENV)
if not VENV.exists():
    run([UV,'venv','--python','3.11','--seed',str(VENV)])

PY = str(VENV / 'bin' / 'python')
print('Voicebox Python:', subprocess.check_output([PY,'--version'], text=True).strip())

MARKER = pathlib.Path('/kaggle/working/.voicebox_epub_kaggle_v1_ready')
if not MARKER.exists():
    run([PY,'-m','pip','install','--upgrade','pip','setuptools','wheel'])
    run([PY,'-m','pip','install','-r',str(ROOT/'backend'/'requirements.txt')])
    run([PY,'-m','pip','install','--no-deps','chatterbox-tts'])
    run([PY,'-m','pip','install','--no-deps','hume-tada'])
    run([PY,'-m','pip','install','git+https://github.com/QwenLM/Qwen3-TTS.git'])
    run([PY,'-c',"import torch, fastapi, uvicorn, fastmcp, httpx, qwen_tts; print('Core imports OK'); print('CUDA in Voicebox env:', torch.cuda.is_available())"])
    MARKER.write_text('ready\n')

print('Voicebox backend installation ready.')


In [ ]:
# Start Voicebox + secure proxy + HTTPS tunnel, then KEEP THIS CELL RUNNING.
import os, subprocess, time, pathlib, secrets, re, urllib.request, html, requests
from IPython.display import display, HTML
from urllib.parse import urlencode

ROOT = pathlib.Path('/kaggle/working/voicebox')
PY = '/kaggle/working/voicebox-py311/bin/python'
if not pathlib.Path(PY).exists():
    raise RuntimeError('Voicebox Python environment is missing. Run the install cell above first.')

for name in ('VOICEBOX_PROCESS','VOICEBOX_PROXY','VOICEBOX_TUNNEL'):
    old = globals().get(name)
    if old is not None:
        try: old.terminate()
        except Exception: pass

token = secrets.token_urlsafe(32)
env = os.environ.copy()
env['VOICEBOX_CORS_ORIGINS'] = 'https://epubplayer-eta.vercel.app'
env['PYTHONUNBUFFERED'] = '1'
voicebox_log_path = pathlib.Path('/kaggle/working/voicebox-backend.log')
voicebox_log = open(voicebox_log_path,'w')
VOICEBOX_PROCESS = subprocess.Popen([PY,'-m','backend.main','--host','127.0.0.1','--port','17493','--data-dir','/kaggle/working/voicebox-data'], cwd=str(ROOT), env=env, stdout=voicebox_log, stderr=subprocess.STDOUT)

for _ in range(180):
    try:
        r = requests.get('http://127.0.0.1:17493/health', timeout=2)
        if r.ok:
            print('Voicebox health:', r.json())
            break
    except Exception:
        pass
    if VOICEBOX_PROCESS.poll() is not None:
        raise RuntimeError('Voicebox failed to start:\n\n' + voicebox_log_path.read_text(errors='replace')[-8000:])
    time.sleep(1)
else:
    raise RuntimeError('Voicebox did not become healthy:\n\n' + voicebox_log_path.read_text(errors='replace')[-8000:])

proxy_code = r'''
import os, httpx
from fastapi import FastAPI, Request
from fastapi.responses import Response
TOKEN = os.environ['VOICEBOX_EPUB_TOKEN']
UPSTREAM = 'http://127.0.0.1:17493'
app = FastAPI()
@app.api_route('/{path:path}', methods=['GET','POST','PUT','DELETE','PATCH','OPTIONS'])
async def proxy(path: str, request: Request):
    if request.method != 'OPTIONS' and request.headers.get('x-voicebox-token') != TOKEN:
        return Response('Unauthorized', status_code=401)
    body = await request.body()
    headers = {k:v for k,v in request.headers.items() if k.lower() not in {'host','content-length','x-voicebox-token','origin','referer'}}
    async with httpx.AsyncClient(timeout=600.0) as client:
        resp = await client.request(request.method, f'{UPSTREAM}/{path}', params=request.query_params, content=body, headers=headers)
    out = {k:v for k,v in resp.headers.items() if k.lower() not in {'content-length','content-encoding','transfer-encoding','connection'}}
    return Response(resp.content, status_code=resp.status_code, headers=out, media_type=resp.headers.get('content-type'))
'''
pathlib.Path('/kaggle/working/voicebox_secure_proxy.py').write_text(proxy_code)
proxy_env = os.environ.copy(); proxy_env['VOICEBOX_EPUB_TOKEN'] = token
proxy_log_path = pathlib.Path('/kaggle/working/voicebox-proxy.log')
proxy_log = open(proxy_log_path,'w')
VOICEBOX_PROXY = subprocess.Popen([PY,'-m','uvicorn','voicebox_secure_proxy:app','--host','127.0.0.1','--port','7860'], cwd='/kaggle/working', env=proxy_env, stdout=proxy_log, stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        if requests.get('http://127.0.0.1:7860/health', headers={'X-Voicebox-Token':token}, timeout=2).ok: break
    except Exception: pass
    if VOICEBOX_PROXY.poll() is not None:
        raise RuntimeError('Secure proxy failed:\n\n' + proxy_log_path.read_text(errors='replace')[-5000:])
    time.sleep(1)
else:
    raise RuntimeError('Secure proxy did not start:\n\n' + proxy_log_path.read_text(errors='replace')[-5000:])

cloudflared = pathlib.Path('/kaggle/working/cloudflared')
if not cloudflared.exists():
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', cloudflared)
    cloudflared.chmod(0o755)
tunnel_log_path = pathlib.Path('/kaggle/working/cloudflared.log')
tunnel_log = open(tunnel_log_path,'w')
VOICEBOX_TUNNEL = subprocess.Popen([str(cloudflared),'tunnel','--url','http://127.0.0.1:7860','--no-autoupdate'], stdout=tunnel_log, stderr=subprocess.STDOUT, text=True)

tunnel_url = None
deadline = time.time() + 60
pattern = re.compile(r'https://[a-zA-Z0-9.-]+\.trycloudflare\.com')
while time.time() < deadline:
    text = tunnel_log_path.read_text(errors='replace') if tunnel_log_path.exists() else ''
    m = pattern.search(text)
    if m:
        tunnel_url = m.group(0); break
    if VOICEBOX_TUNNEL.poll() is not None:
        raise RuntimeError('Cloudflare tunnel exited:\n\n' + text[-5000:])
    time.sleep(.5)
if not tunnel_url:
    raise RuntimeError('Cloudflare tunnel did not provide a URL.\n\n' + tunnel_log_path.read_text(errors='replace')[-5000:])

pair = 'https://epubplayer-eta.vercel.app/app/settings#' + urlencode({'voiceboxUrl': tunnel_url, 'voiceboxToken': token})
print('VOICEBOX SERVER:', tunnel_url)
print('ACCESS TOKEN:', token)
display(HTML(f'''<div style="font-family:-apple-system;padding:18px;border:1px solid #ddd;border-radius:14px"><h2>Voicebox is ready</h2><p>This Kaggle cell will stay running in the cloud. You can switch to EPUB Player now.</p><a href="{html.escape(pair)}" target="_blank" style="display:inline-block;padding:14px 18px;background:#6d5dfc;color:white;text-decoration:none;border-radius:12px;font-weight:700">Connect EPUB Player to Voicebox</a></div>'''))

print('\nServer heartbeat started. Leave this cell running while you listen.')
heartbeat = 0
while True:
    if VOICEBOX_PROCESS.poll() is not None:
        raise RuntimeError('Voicebox backend stopped:\n\n' + voicebox_log_path.read_text(errors='replace')[-8000:])
    if VOICEBOX_PROXY.poll() is not None:
        raise RuntimeError('Voicebox proxy stopped:\n\n' + proxy_log_path.read_text(errors='replace')[-5000:])
    if VOICEBOX_TUNNEL.poll() is not None:
        raise RuntimeError('Cloudflare tunnel stopped:\n\n' + tunnel_log_path.read_text(errors='replace')[-5000:])
    try:
        ok = requests.get('http://127.0.0.1:7860/health', headers={'X-Voicebox-Token':token}, timeout=5).ok
    except Exception:
        ok = False
    heartbeat += 1
    if heartbeat % 6 == 0:
        print(time.strftime('%H:%M:%S'), 'Voicebox server alive' if ok else 'Voicebox health check failed', flush=True)
    time.sleep(10)
